# EMCAD ClinicDB Replication — Notebook 1: Environment Preparation

Kaggle settings:

- Accelerator: **None**
- Internet: **On**
- Input: **cvc-clinicdb-raw**

This notebook prepares the exact replication environment and data layout. It does **not** train EMCAD.

In [1]:
!python --version

Python 3.12.13


In [2]:
from pathlib import Path

input_root=Path("/kaggle/input")
pairs=[]

for image_root in input_root.rglob("Original"):
    if not image_root.is_dir():
        continue
    mask_root=image_root.parent/"Ground Truth"
    if not mask_root.is_dir():
        continue

    images=[p for p in image_root.iterdir() if p.is_file() and p.suffix.lower() in {".tif",".tiff"}]
    masks=[p for p in mask_root.iterdir() if p.is_file() and p.suffix.lower() in {".tif",".tiff"}]

    if len(images)==612 and len(masks)==612:
        pairs.append((image_root,mask_root))

assert len(pairs)==1,f"Expected one ClinicDB dataset, found {len(pairs)}"

source_images,source_masks=pairs[0]

image_ids={int(p.stem) for p in source_images.iterdir() if p.is_file() and p.suffix.lower() in {".tif",".tiff"}}
mask_ids={int(p.stem) for p in source_masks.iterdir() if p.is_file() and p.suffix.lower() in {".tif",".tiff"}}

assert image_ids==set(range(1,613))
assert mask_ids==set(range(1,613))

print("ClinicDB input verified")
print("Images:",source_images)
print("Masks:",source_masks)
print("Count:",len(image_ids),len(mask_ids))


ClinicDB input verified
Images: /kaggle/input/datasets/anirbanroysudipto/cvc-clinicdb-raw/CVC-ClinicDB/Original
Masks: /kaggle/input/datasets/anirbanroysudipto/cvc-clinicdb-raw/CVC-ClinicDB/Ground Truth
Count: 612 612


In [3]:
%%bash
set -euo pipefail

cd /kaggle/working
rm -rf EMCAD

git clone https://github.com/SLDGroup/EMCAD.git
cd EMCAD
git checkout 26c9c31f731f749b62c5fe83f44376dac75f3aa8

git rev-parse HEAD | tee /kaggle/working/EMCAD_AUTHOR_COMMIT.txt
git diff --exit-code
git diff --cached --exit-code
git status --porcelain --untracked-files=no


26c9c31f731f749b62c5fe83f44376dac75f3aa8


Cloning into 'EMCAD'...
Note: switching to '26c9c31f731f749b62c5fe83f44376dac75f3aa8'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 26c9c31 adding trained weights on Synapse dataset


In [4]:
from pathlib import Path
import csv
import cv2
import numpy as np
import shutil

train_ids=[1, 2, 6, 7, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 30, 31, 32, 35, 37, 38, 39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 61, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 124, 126, 127, 130, 131, 132, 133, 134, 135, 136, 137, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 157, 158, 160, 161, 162, 164, 165, 166, 168, 170, 172, 174, 175, 176, 177, 178, 179, 180, 183, 184, 185, 186, 187, 188, 189, 190, 192, 194, 195, 196, 201, 202, 204, 205, 206, 207, 208, 209, 210, 211, 213, 215, 216, 217, 219, 221, 224, 226, 227, 230, 231, 233, 234, 235, 236, 237, 238, 239, 240, 242, 243, 245, 246, 247, 249, 250, 251, 252, 253, 255, 256, 257, 258, 259, 260, 261, 264, 265, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 280, 281, 282, 283, 284, 285, 286, 288, 289, 290, 291, 292, 293, 294, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 308, 309, 310, 311, 313, 315, 316, 319, 320, 322, 323, 324, 325, 326, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 341, 342, 344, 345, 346, 347, 349, 351, 354, 356, 357, 358, 361, 362, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 391, 393, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 417, 419, 420, 421, 422, 423, 424, 426, 427, 428, 429, 430, 431, 432, 434, 435, 436, 438, 439, 442, 444, 445, 446, 447, 448, 449, 450, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 469, 470, 472, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 489, 490, 492, 493, 495, 496, 498, 499, 500, 501, 502, 503, 504, 508, 509, 510, 511, 512, 515, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 529, 530, 532, 533, 534, 535, 536, 537, 538, 540, 541, 542, 543, 544, 545, 546, 547, 549, 550, 551, 553, 554, 556, 557, 558, 559, 560, 561, 562, 563, 564, 566, 567, 570, 571, 573, 574, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 589, 592, 595, 596, 597, 598, 599, 600, 601, 602, 603, 604, 605, 606, 607, 609, 610, 611, 612]
val_ids=[3, 4, 10, 33, 34, 36, 60, 80, 81, 109, 163, 173, 181, 191, 197, 198, 200, 203, 214, 218, 228, 229, 241, 244, 254, 262, 266, 279, 295, 307, 312, 314, 318, 343, 348, 350, 353, 355, 364, 392, 418, 433, 437, 443, 471, 473, 486, 491, 497, 506, 507, 514, 516, 531, 548, 552, 555, 565, 568, 569, 576]
test_ids=[5, 8, 27, 29, 40, 59, 64, 89, 117, 125, 128, 129, 139, 156, 159, 167, 169, 171, 182, 193, 199, 212, 220, 222, 223, 225, 232, 248, 263, 287, 317, 321, 327, 328, 340, 352, 359, 360, 363, 378, 390, 394, 416, 425, 440, 441, 451, 468, 487, 488, 494, 505, 513, 528, 539, 572, 575, 590, 591, 593, 594, 608]

splits={"train":train_ids,"val":val_ids,"test":test_ids}

assert len(train_ids)==489
assert len(val_ids)==61
assert len(test_ids)==62
assert len(set(train_ids)|set(val_ids)|set(test_ids))==612
assert not(set(train_ids)&set(val_ids))
assert not(set(train_ids)&set(test_ids))
assert not(set(val_ids)&set(test_ids))

def build_map(folder):
    return {int(p.stem):p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {".tif",".tiff"}}

image_map=build_map(source_images)
mask_map=build_map(source_masks)

target=Path("/kaggle/working/EMCAD/data/polyp/target/ClinicDB")
if target.exists():
    shutil.rmtree(target)

manifest=[]

for split,ids in splits.items():
    image_dir=target/split/"images"
    mask_dir=target/split/"masks"
    image_dir.mkdir(parents=True,exist_ok=True)
    mask_dir.mkdir(parents=True,exist_ok=True)

    for frame_id in ids:
        src_image=image_map[frame_id]
        src_mask=mask_map[frame_id]
        dst_image=image_dir/f"{frame_id}.png"
        dst_mask=mask_dir/f"{frame_id}.png"

        image=cv2.imread(str(src_image),cv2.IMREAD_COLOR)
        mask=cv2.imread(str(src_mask),cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Could not decode {src_image}")
        if mask is None:
            raise RuntimeError(f"Could not decode {src_mask}")

        assert image.shape==(288,384,3),(src_image,image.shape)
        assert mask.shape==(288,384),(src_mask,mask.shape)
        assert image.dtype==np.uint8,(src_image,image.dtype)
        assert mask.dtype==np.uint8,(src_mask,mask.dtype)

        assert cv2.imwrite(str(dst_image),image)
        assert cv2.imwrite(str(dst_mask),mask)

        check_image=cv2.imread(str(dst_image),cv2.IMREAD_COLOR)
        check_mask=cv2.imread(str(dst_mask),cv2.IMREAD_GRAYSCALE)

        assert np.array_equal(image,check_image),frame_id
        assert np.array_equal(mask,check_mask),frame_id

        manifest.append((frame_id,src_image.name,dst_image.name,split))

manifest_path=Path("/kaggle/working/ClinicDB_author_split_manifest.csv")
with manifest_path.open("w",newline="") as f:
    writer=csv.writer(f)
    writer.writerow(["frame_id","source_filename","staged_filename","split"])
    writer.writerows(manifest)

for split,expected in [("train",489),("val",61),("test",62)]:
    images=sorted((target/split/"images").glob("*.png"),key=lambda p:int(p.stem))
    masks=sorted((target/split/"masks").glob("*.png"),key=lambda p:int(p.stem))

    assert len(images)==expected
    assert len(masks)==expected
    assert [p.name for p in images]==[p.name for p in masks]

    print(split,len(images),len(masks))

print("ClinicDB staging verified")
print("Manifest:",manifest_path)


[ WARN:0@0.315] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
[ WARN:0@0.363] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
[ WARN:0@0.387] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
[ WARN:0@0.413] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
[ WARN:0@0.441] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Sum of Photometric type-related color channels and ExtraSamples doesn'

train 489 489
val 61 61
test 62 62
ClinicDB staging verified
Manifest: /kaggle/working/ClinicDB_author_split_manifest.csv


[ WARN:0@15.588] global grfmt_tiff.cpp:122 TIFF_Warning TIFFReadDirectory: Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


In [5]:
%%bash
set -euo pipefail

rm -rf /kaggle/working/miniforge /kaggle/working/emcadenv

curl -fL --retry 8 --retry-all-errors --retry-delay 5 --connect-timeout 30 https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -o /tmp/miniforge.sh

bash /tmp/miniforge.sh -b -p /kaggle/working/miniforge

/kaggle/working/miniforge/bin/conda create -y -p /kaggle/working/emcadenv python=3.8 pip


PREFIX=/kaggle/working/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.7.22-hbd8a1cb_0.conda
Extracting libgomp-16.2.0-he0feb66_4.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-9_cp314.conda
Extracting tzdata-2026c-h151e31d_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_102.conda
Extracting libgcc-16.2.0-ha9f2e26_4.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.8-hebe6cf0_2.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.8.1-hecca717_1.conda
Extracting libffi-3.7.0-h81df57d_1.conda
Extracting libiconv-1.18-h0cb94f2_3.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extracting libstdcxx-16.2.0-h934c35e_4.co

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  118M  100  118M    0     0   127M      0 --:--:-- --:--:-- --:--:--  367M
warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.
WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



In [6]:
%%bash
set -euo pipefail

ENV=/kaggle/working/emcadenv
cd /kaggle/working/EMCAD

$ENV/bin/python -m pip install --upgrade "pip<25"

$ENV/bin/pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113

$ENV/bin/pip install mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.11.0/index.html

$ENV/bin/pip install -r requirements.txt
$ENV/bin/pip install --force-reinstall --no-deps ptflops==0.7.2.2
$ENV/bin/pip install openpyxl


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu113
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 56.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 124.5 MB/s eta 0:00:00
Looking in links: https://download.openmmlab.com/mmcv/dist/cu113/torch1.11.0/index.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 21.0 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.1 MB/s eta 0:00:00
  Installin

In [7]:
%%bash
set -euo pipefail

cd /kaggle/working/EMCAD
mkdir -p pretrained_pth/pvt

curl -fL --retry 8 --retry-all-errors --retry-delay 5 --connect-timeout 30 https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b2.pth -o pretrained_pth/pvt/pvt_v2_b2.pth

test $(stat -c%s pretrained_pth/pvt/pvt_v2_b2.pth) -gt 50000000
sha256sum pretrained_pth/pvt/pvt_v2_b2.pth | tee /kaggle/working/PVT_V2_B2_SHA256.txt
ls -lh pretrained_pth/pvt/pvt_v2_b2.pth


80711cd1b37ffba12bec6c7a2a7c54efe2315ed635e6d2055fd51c3e909ede4d  pretrained_pth/pvt/pvt_v2_b2.pth
-rw-r--r-- 1 root root 97M Sep 26 21:00 pretrained_pth/pvt/pvt_v2_b2.pth


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 96.7M  100 96.7M    0     0  21.9M      0  0:00:04  0:00:04 --:--:-- 26.9M


In [8]:
%%bash
set -euo pipefail

ENV=/kaggle/working/emcadenv
cd /kaggle/working/EMCAD

$ENV/bin/python - <<'PY'
import sys
import torch
import numpy
import timm
import ptflops

from ptflops import get_model_complexity_info
from utils.utils import cal_params_flops
from utils.dataloader_polyp import get_loader
from lib.networks import EMCADNet

print("Python:",sys.version.split()[0])
print("PyTorch:",torch.__version__)
print("CUDA build:",torch.version.cuda)
print("CUDA available:",torch.cuda.is_available())
print("NumPy:",numpy.__version__)
print("timm:",timm.__version__)
print("ptflops:",ptflops.__version__ if hasattr(ptflops,"__version__") else "0.7.2.2")

assert sys.version_info[:2]==(3,8)
assert torch.__version__.startswith("1.11.0")

loader=get_loader(
    image_root="./data/polyp/target/ClinicDB/train/images/",
    gt_root="./data/polyp/target/ClinicDB/train/masks/",
    batchsize=2,
    trainsize=352,
    shuffle=False,
    augmentation=True,
    split="train",
    color_image=True
)

images,masks=next(iter(loader))

assert tuple(images.shape)==(2,3,352,352)
assert tuple(masks.shape)==(2,1,352,352)

model=EMCADNet(
    num_classes=1,
    kernel_sizes=[1,3,5],
    expansion_factor=2,
    dw_parallel=True,
    add=True,
    lgag_ks=3,
    activation="relu6",
    encoder="pvt_v2_b2",
    pretrain=True,
    pretrained_dir="./pretrained_pth/pvt/"
)

print("ptflops import: OK")
print("utils.utils import: OK")
print("Author dataloader: OK")
print("EMCAD model construction: OK")
PY


Python: 3.8.20
PyTorch: 1.11.0+cu113
CUDA build: 11.3
CUDA available: False
NumPy: 1.22.4
timm: 0.6.12
ptflops: 0.7.2.2
Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515
ptflops import: OK
utils.utils import: OK
Author dataloader: OK
EMCAD model construction: OK


In [9]:
%%bash
set -euo pipefail

cd /kaggle/working/EMCAD

git diff --exit-code
git diff --cached --exit-code
git status --porcelain --untracked-files=no

echo "Tracked EMCAD source files are unchanged."


Tracked EMCAD source files are unchanged.


In [10]:
%%bash
set -euo pipefail

ENV=/kaggle/working/emcadenv
MINIFORGE=/kaggle/working/miniforge

$ENV/bin/pip freeze | sort > /kaggle/working/EMCAD_ENVIRONMENT.txt

$MINIFORGE/bin/python -m pip install -q conda-pack

rm -f /kaggle/working/emcadenv.tar.gz
$MINIFORGE/bin/conda-pack -p /kaggle/working/emcadenv -o /kaggle/working/emcadenv.tar.gz

cd /kaggle/working
rm -f EMCAD_prepared.tar.gz
tar -czf EMCAD_prepared.tar.gz EMCAD

gzip -t emcadenv.tar.gz
gzip -t EMCAD_prepared.tar.gz

sha256sum emcadenv.tar.gz EMCAD_prepared.tar.gz EMCAD_AUTHOR_COMMIT.txt ClinicDB_author_split_manifest.csv EMCAD_ENVIRONMENT.txt PVT_V2_B2_SHA256.txt > EMCAD_NOTEBOOK1_SHA256SUMS.txt


Packing environment at '/kaggle/working/emcadenv' to '/kaggle/working/emcadenv.tar.gz'
[########################################] | 100% Completed |  3min 40.8s


In [11]:
from pathlib import Path

working=Path("/kaggle/working")

required=[
    "emcadenv.tar.gz",
    "EMCAD_prepared.tar.gz",
    "EMCAD_AUTHOR_COMMIT.txt",
    "ClinicDB_author_split_manifest.csv",
    "EMCAD_ENVIRONMENT.txt",
    "PVT_V2_B2_SHA256.txt",
    "EMCAD_NOTEBOOK1_SHA256SUMS.txt"
]

for name in required:
    path=working/name
    assert path.is_file(),path
    print(f"{name}: {path.stat().st_size/1024**2:.2f} MB")

commit=(working/"EMCAD_AUTHOR_COMMIT.txt").read_text().strip()
assert commit=="26c9c31f731f749b62c5fe83f44376dac75f3aa8"

print()
print("Notebook 1 preparation complete")
print("Author commit:",commit)
print("Save this notebook as a Kaggle Version with outputs.")


emcadenv.tar.gz: 2058.04 MB
EMCAD_prepared.tar.gz: 143.81 MB
EMCAD_AUTHOR_COMMIT.txt: 0.00 MB
ClinicDB_author_split_manifest.csv: 0.02 MB
EMCAD_ENVIRONMENT.txt: 0.00 MB
PVT_V2_B2_SHA256.txt: 0.00 MB
EMCAD_NOTEBOOK1_SHA256SUMS.txt: 0.00 MB

Notebook 1 preparation complete
Author commit: 26c9c31f731f749b62c5fe83f44376dac75f3aa8
Save this notebook as a Kaggle Version with outputs.
